# More levers: test-time averaging, balanced sampling, ensembles, Wikidata-only labels

Before you start: **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. About **1.5 hours** on a T4 (Part A about 30 minutes, Part B about 1 hour, Part C a few minutes).

This notebook runs only the items from the improvement list that had **not** been tested yet. Everything else is already in the repo:

| Item | Status before this notebook |
|---|---|
| 1. Stronger pretrained model | Done: CLIP, DINOv2, SigLIP probes, CLIP fine-tune at lr 1e-5 (`results/improve/`) |
| 2. Rare styles | Class-weighted loss is already in every model. **New here: balanced sampling** (Part B) |
| 3. Augmentation | Random resized crop and horizontal flip are in the baseline; colour jitter tested with 3 seeds (`results/colorjitter/`) |
| 4. Resolution and test-time averaging | The images are stored at 256 px, so training at 384 would only upscale them. **New here: 7-view test-time averaging for the CLIP and SigLIP probes** (Part A) |
| 5. Label check | Label source analysis is in `results/final/label_source.csv`. **New here: Wikidata-labelled paintings scored separately for every model** (Part C) |
| 6. Ensembles | Seed and ResNet + CLIP + DINOv2 ensembles are in `results/ensembles/`. **New here: ensembles with the SigLIP and balanced-sampling models** (Part C) |

Rules as before: same artist-disjoint split and museum test set; main numbers are for **painters not in WikiArt train**; every setting is chosen on **WikiArt val only**; intervals resample **painters**, not single paintings.

**If Colab disconnects:** just **Run all** again. Finished features and seeds are on Drive and are skipped.

## 0. Settings (leave as they are)

In [ ]:
RUN_TTA = True        # Part A: test-time averaging for the CLIP and SigLIP probes (~30 min)
RUN_BALANCED = True   # Part B: ResNet-50 with balanced sampling, 3 seeds (~20 min each)
SEEDS = [1, 2, 3]
import warnings; warnings.filterwarnings('ignore')

## 1. Check for a GPU, connect Drive, unpack the data, get the repository (about 3 minutes)

In [ ]:
import torch, os, time, shutil, subprocess, glob, zipfile
T_START = time.time()
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Go to Runtime -> Change runtime type -> pick T4 GPU -> Save, then Runtime -> Run all again. '
                       '(If T4 is greyed out, your free GPU time is used up for today: try again later or use your other account.)')
print('GPU:', torch.cuda.get_device_name(0))

def ensure_data():
    # Mount Drive, find colab_data.zip (My Drive, the shared-folder shortcut, or one folder deeper) and unpack it into /content.
    global BASE, OUT
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    found = (glob.glob('/content/drive/MyDrive/wikiart-to-museum/colab_data.zip') + glob.glob('/content/drive/MyDrive/colab_data.zip')
             + glob.glob('/content/drive/MyDrive/*/colab_data.zip') + glob.glob('/content/drive/MyDrive/*/*/colab_data.zip'))
    if not found:
        raise RuntimeError('colab_data.zip not found on Drive. In Google Drive, add the shared wikiart-to-museum folder to My Drive '
                           '(right-click the folder -> Organize -> Add shortcut -> My Drive), then Runtime -> Run all again.')
    BASE = os.path.dirname(found[0])
    OUT = f'{BASE}/wikiart_runs/more_levers'; os.makedirs(OUT, exist_ok=True)
    if not os.path.exists('/content/data/final/wikiart_split.csv'):
        print('Copying and unpacking', found[0], '(about 3 minutes)...', flush=True)
        shutil.copy(found[0], '/content/colab_data.zip')
        with zipfile.ZipFile('/content/colab_data.zip') as z: z.extractall('/content')
        os.remove('/content/colab_data.zip')
    if not os.path.exists('/content/data/final/wikiart_split.csv'):
        hits = glob.glob('/content/**/final/wikiart_split.csv', recursive=True)
        if hits:   # zip unpacked one folder deeper: link it to /content/data
            os.symlink(os.path.dirname(os.path.dirname(hits[0])), '/content/data')
    if not os.path.exists('/content/data/final/wikiart_split.csv'):
        raise RuntimeError('Unpacked colab_data.zip but data/final/wikiart_split.csv is missing. Runtime -> Disconnect and delete runtime, then Run all again.')
    print('Data :', found[0]); print('Saves:', OUT)

ensure_data()
# the repository: train.py with the --balanced-sampler switch, and the saved baseline predictions
if not os.path.exists('/content/repo/src/train.py'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/abdull254/wikiart-to-museum', '/content/repo'], check=True)
assert '--balanced-sampler' in open('/content/repo/src/train.py').read(), 'repo train.py has no --balanced-sampler switch'
shutil.copy('/content/repo/src/train.py', '/content/train_bs.py')
print('repository ready')

## 2. Sets, painters and scoring (same as the table of record)

In [ ]:
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.metrics import balanced_accuracy_score as BA, f1_score
ensure_data()   # in case the runtime was reset after step 1
DATA = Path('/content/data'); REPO = Path('/content/repo')
split = pd.read_csv(DATA / 'final/wikiart_split.csv')
CLASSES = sorted(split['style'].unique()); PC = [f'p_{c}' for c in CLASSES]
museum = pd.read_csv(DATA / 'final/museum_test.csv'); museum = museum[museum['style'].isin(CLASSES)]
sets = {s: split[split['split'] == s] for s in ('train', 'val', 'test')}
sets['met'] = museum[museum['museum'] == 'met']; sets['rijks'] = museum[museum['museum'] == 'rijks']
sets = {k: df[df['image'].map(lambda p: (DATA / p).exists())].reset_index(drop=True) for k, df in sets.items()}
Y = {k: df['style'].map(CLASSES.index).values for k, df in sets.items()}
for k, df in sets.items(): print(f'{k:<6} {len(df):>6}')
print('Expected: train 18501, val 3786, test 3857, met 937, rijks 3092')

# painter of every painting (anonymous paintings are their own cluster), as in src/final_analysis.py
def painter(k):
    df = sets[k]
    if k in ('train', 'val', 'test'): return ('w:' + df['artist_key'].astype(str)).values
    anon = df['artist'].map(lambda a: (not isinstance(a, str)) or a.strip().lower() in ('', 'unknown', 'anonymous', 'anoniem', 'onbekend'))
    key = df['museum'] + ':' + df['id'].astype(str)
    return np.where(anon, 'anon:' + key, 'm:' + df['artist'].astype(str))
def mask(k, group='unseen'):
    if k in ('train', 'val', 'test'): return np.ones(len(sets[k]), bool)
    m = sets[k]['artist_group'].values == 'unseen'
    if group == 'wikidata': m &= sets[k]['source'].values == 'artwork movement (Wikidata)'
    return m
rng = np.random.default_rng(0)
def boot(y, preds, painters, B=1000):
    """painter-level bootstrap of balanced accuracy for several prediction vectors (same resamples)"""
    u, inv = np.unique(painters, return_inverse=True); groups = [np.where(inv == j)[0] for j in range(len(u))]
    out = np.empty((B, len(preds)))
    for b in range(B):
        idx = np.concatenate([groups[j] for j in rng.integers(0, len(groups), len(groups))])
        out[b] = [BA(y[idx], p[idx]) for p in preds]
    return 100 * out
def metrics(y, P):
    p = P.argmax(1)
    return dict(bal_acc=100 * BA(y, p), top1=100 * (y == p).mean(),
                top3=100 * (np.argsort(-P, 1)[:, :3] == y[:, None]).any(1).mean(), macro_f1=100 * f1_score(y, p, average='macro'))
REPORT = []
def say(s=''):
    print(s); REPORT.append(str(s))
def save_preds(name, k, P):
    d = sets[k][['image', 'style']].copy(); d['pred'] = [CLASSES[i] for i in P.argmax(1)]
    for i, c in enumerate(CLASSES): d[f'p_{c}'] = P[:, i]
    if 'artist_group' in sets[k]: d['artist_group'] = sets[k]['artist_group'].values
    d.to_csv(f'{OUT}/preds_{name}_{k}.csv', index=False)

## Part A. Test-time averaging for the CLIP and SigLIP probes (about 30 minutes)
Each test painting is scored from 7 views (whole painting, its mirror image, 4 corner crops and the centre crop) and the 7 predictions are averaged. Training uses the whole painting only, exactly as before. Features are saved to Drive, so this runs only once.

In [ ]:
if RUN_TTA:
    from PIL import Image
    from torch.utils.data import Dataset, DataLoader
    from torchvision import transforms as TT
    from transformers import CLIPModel, AutoModel
    dev = 'cuda'
    def vec(o): return o if torch.is_tensor(o) else o.pooler_output
    SPECS = {'clip':   (['openai/clip-vit-large-patch14'], (0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)),
             'siglip': (['google/siglip-so400m-patch14-224', 'google/siglip-so400m-patch14-384'], (0.5, 0.5, 0.5), (0.5, 0.5, 0.5))}
    class Views(Dataset):
        def __init__(self, paths, size, mean, std, n_views):
            self.paths, self.size, self.n = paths, size, n_views
            self.norm = TT.Compose([TT.ToTensor(), TT.Normalize(mean, std)])
        def __len__(self): return len(self.paths)
        def __getitem__(self, i):
            img = Image.open(DATA / self.paths[i]).convert('RGB')
            whole = img.resize((self.size, self.size), Image.BICUBIC)
            if self.n == 1: return self.norm(whole)[None]
            big = img.resize((round(self.size * 256 / 224),) * 2, Image.BICUBIC)
            return torch.stack([self.norm(v) for v in (whole, TT.functional.hflip(whole), *TT.FiveCrop(self.size)(big))])

    @torch.no_grad()
    def extract(name):
        f = f'{OUT}/tta_features_{name}.npz'
        if os.path.exists(f):
            F = dict(np.load(f))
            if all(k in F and len(F[k]) == len(sets[k]) for k in sets): print(name, 'features already on Drive'); return F
        ids, mean, std = SPECS[name]
        for mid in ids:
            try:
                model = (CLIPModel if name == 'clip' else AutoModel).from_pretrained(mid, torch_dtype=torch.float16).to(dev).eval(); break
            except Exception as e:
                print('could not load', mid, '->', e)
        size = model.config.vision_config.image_size; print(name, 'model', mid, 'at', size, 'px')
        F, t0 = {}, time.time()
        for k, df in sets.items():
            nv = 1 if k == 'train' else 7
            dl = DataLoader(Views(df['image'].tolist(), size, mean, std, nv), batch_size=128 // nv if nv > 1 else 128, num_workers=2)
            out = []
            for x in dl:
                b = x.shape[0]
                out.append(vec(model.get_image_features(pixel_values=x.flatten(0, 1).to(dev, torch.float16))).float().view(b, nv, -1).cpu().numpy())
            F[k] = np.concatenate(out)
            print(f'  {name} {k:<6} {F[k].shape}  ({(time.time()-t0)/60:.1f} min)', flush=True)
        np.savez(f, **F); del model; torch.cuda.empty_cache()
        return F

    FT = {n: extract(n) for n in SPECS}
    print(f'features done, elapsed {(time.time()-T_START)/60:.0f} min')

In [ ]:
if RUN_TTA:
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    TTA = {}   # name -> {set: (single-view probs, 7-view averaged probs)}
    def fit_probe(names):
        def feats(k):   # (n, views, d), each model's features L2-normalised, then joined
            return np.concatenate([FT[n][k] / np.linalg.norm(FT[n][k], axis=-1, keepdims=True) for n in names], -1)
        tr = feats('train')[:, 0]; sc = StandardScaler().fit(tr); tr = sc.transform(tr)
        ev = {k: feats(k) for k in ('val', 'test', 'met', 'rijks')}
        best = None
        for C in (0.003, 0.01, 0.03, 0.1):
            clf = LogisticRegression(C=C, max_iter=3000, class_weight='balanced').fit(tr, Y['train'])
            b = BA(Y['val'], clf.predict(sc.transform(ev['val'][:, 0])))
            print(f'  {"+".join(names):<12} C={C:<6} val bal acc {b:.3f}', flush=True)
            if best is None or b > best[0]: best = (b, clf)
        out = {}
        for k, X in ev.items():
            n, v, d = X.shape
            P = best[1].predict_proba(sc.transform(X.reshape(n * v, d))).reshape(n, v, -1)
            out[k] = (P[:, 0], P.mean(1))
        return out
    for names in (['clip'], ['siglip'], ['clip', 'siglip']):
        nm = '+'.join(names); TTA[nm] = fit_probe(names)
        for k in ('val', 'test', 'met', 'rijks'):
            save_preds(f'{nm}_probe', k, TTA[nm][k][0]); save_preds(f'{nm}_probe_tta', k, TTA[nm][k][1])

    say('PART A. TEST-TIME AVERAGING (7 views) for linear probes; museum = painters not in WikiArt train')
    say('single = whole painting only; tta = mean of 7 views; diff = tta - single, painter-level paired bootstrap\n')
    rows = []
    for nm, R in TTA.items():
        for k in ('val', 'test', 'met', 'rijks'):
            m = mask(k); y = Y[k][m]; P1, P7 = R[k][0][m], R[k][1][m]
            a, b = metrics(y, P1), metrics(y, P7)
            r = dict(probe=nm, set=k, n=int(m.sum()), single_bal=a['bal_acc'], tta_bal=b['bal_acc'], diff=b['bal_acc'] - a['bal_acc'],
                     single_top1=a['top1'], tta_top1=b['top1'], tta_top3=b['top3'])
            if k != 'val':
                bs = boot(y, [P7.argmax(1), P1.argmax(1)], painter(k)[m], B=500); d = bs[:, 0] - bs[:, 1]
                r.update(ci_low=np.percentile(d, 2.5), ci_high=np.percentile(d, 97.5), p=min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())))
            rows.append(r)
    tta_tab = pd.DataFrame(rows).round(2); tta_tab.to_csv(f'{OUT}/tta_summary.csv', index=False)
    pd.set_option('display.width', 250); say(tta_tab.to_string(index=False))
    open(f'{OUT}/more_levers_report.txt', 'w').write('\n'.join(REPORT))
    print(f'\nelapsed {(time.time()-T_START)/60:.0f} min')

## Part B. ResNet-50 with balanced sampling (about 20 minutes per seed)
The only change from the baseline: training batches are drawn so that **every style is equally likely** (`src/train.py --balanced-sampler`), and the loss is then unweighted (the baseline uses a class-weighted loss instead; using both would count the rare styles twice). Same 12 epochs, best epoch on WikiArt val, seeds 1-3.

In [ ]:
def finished(d):
    return os.path.exists(f'{d}/metrics.json') and os.path.exists(f'{d}/best.pt') and os.path.getsize(f'{d}/best.pt') > 10_000_000
if RUN_BALANCED:
    %cd /content
    for SEED in SEEDS:
        name = f'resnet50_bs_seed{SEED}'; dst = f'{OUT}/{name}'
        if finished(dst): print(f'seed {SEED}: already on Drive, skipping'); continue
        t0 = time.time(); print(f'\n===== seed {SEED}: balanced sampling =====', flush=True)
        if os.path.exists(f'/content/runs/{name}'): shutil.rmtree(f'/content/runs/{name}')
        !python /content/train_bs.py --seed {SEED} --balanced-sampler --eval-museums --workers 2 --name {name}
        if not finished(f'/content/runs/{name}'): raise SystemExit(f'seed {SEED} did not finish. Run all again to retry it.')
        if os.path.exists(dst): shutil.move(dst, dst + time.strftime('_old_%Y%m%d_%H%M%S'))
        shutil.copytree(f'/content/runs/{name}', dst); assert finished(dst), 'copy to Drive failed: check Drive space'
        print(f'seed {SEED}: saved to Drive in {(time.time()-t0)/60:.0f} min')

In [ ]:
# compare with the 3 committed baseline seeds on the same paintings
KEYSET = {'test': 'wikiart_test', 'met': 'met', 'rijks': 'rijks'}
def run_probs(run_dir, k):
    d = pd.read_csv(f'{run_dir}/preds_{KEYSET[k]}.csv', dtype={'id': str}).set_index('id')
    key = sets[k]['wikiart_idx'].astype(str) if k == 'test' else sets[k]['museum'] + ':' + sets[k]['id'].astype(str)
    P = d.loc[key.values, PC].values.astype(float); return P / P.sum(1, keepdims=True)
BASE_RUNS = [REPO / f'results/runs/resnet50_seed{s}' for s in (1, 2, 3)]
BS_RUNS = [f'{OUT}/resnet50_bs_seed{s}' for s in SEEDS if finished(f'{OUT}/resnet50_bs_seed{s}')]
if BS_RUNS:
    say(f'\nPART B. BALANCED SAMPLING vs BASELINE (class-weighted loss), ResNet-50, {len(BS_RUNS)} seed(s) vs 3 baseline seeds')
    say('bal acc = mean over seeds; diff = balanced sampling - baseline; painter-level bootstrap of the seed-mean difference\n')
    rows, rec = [], []
    for k in ('test', 'met', 'rijks'):
        m = mask(k); y = Y[k][m]
        PB = [run_probs(r, k)[m] for r in BASE_RUNS]; PS = [run_probs(r, k)[m] for r in BS_RUNS]
        bb = boot(y, [p.argmax(1) for p in PB + PS], painter(k)[m], B=500)
        d = bb[:, 3:].mean(1) - bb[:, :3].mean(1)
        mb = np.mean([BA(y, p.argmax(1)) for p in PB]) * 100; ms = np.mean([BA(y, p.argmax(1)) for p in PS]) * 100
        rows.append(dict(set=k, n=int(m.sum()), baseline_bal=mb, balanced_bal=ms, diff=ms - mb,
                         ci_low=np.percentile(d, 2.5), ci_high=np.percentile(d, 97.5), p=min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())),
                         baseline_top1=np.mean([(y == p.argmax(1)).mean() for p in PB]) * 100,
                         balanced_top1=np.mean([(y == p.argmax(1)).mean() for p in PS]) * 100))
        for i, c in enumerate(CLASSES):
            rb = np.mean([(p.argmax(1)[y == i] == i).mean() for p in PB]) * 100; rs = np.mean([(p.argmax(1)[y == i] == i).mean() for p in PS]) * 100
            rec.append(dict(set=k, style=c, baseline_recall=rb, balanced_recall=rs, diff=rs - rb))
    bs_tab = pd.DataFrame(rows).round(2); bs_rec = pd.DataFrame(rec).round(1)
    bs_tab.to_csv(f'{OUT}/balanced_summary.csv', index=False); bs_rec.to_csv(f'{OUT}/balanced_recall.csv', index=False)
    say(bs_tab.to_string(index=False)); say('\nper-style recall'); say(bs_rec.to_string(index=False))
    open(f'{OUT}/more_levers_report.txt', 'w').write('\n'.join(REPORT))

## Part C. Ensembles and Wikidata-only labels (a few minutes, no GPU needed)
Equal-weight averages of class probabilities; nothing is tuned. Every model is also scored on the museum paintings whose style comes **directly from Wikidata** (the artwork's own movement). There are only about 50 such unseen-painter paintings at the Met and none at the Rijksmuseum, so that row is a check, not a headline.

In [ ]:
MEMBERS = {'ResNet-50 (3 seeds)': lambda k: np.mean([run_probs(r, k) for r in BASE_RUNS], 0)}
if BS_RUNS: MEMBERS['ResNet-50 balanced (seeds)'] = lambda k: np.mean([run_probs(r, k) for r in BS_RUNS], 0)
def from_csv(name):
    def f(k):
        d = pd.read_csv(f'{OUT}/preds_{name}_{k}.csv'); assert (d['image'].values == sets[k]['image'].values).all(); return d[PC].values
    return f
for nm, label in (('clip_probe_tta', 'CLIP probe (TTA)'), ('siglip_probe_tta', 'SigLIP probe (TTA)'), ('clip+siglip_probe_tta', 'CLIP+SigLIP probe (TTA)')):
    if os.path.exists(f'{OUT}/preds_{nm}_met.csv'): MEMBERS[label] = from_csv(nm)
ENS = {}
if 'SigLIP probe (TTA)' in MEMBERS:
    ENS['ResNet-50 + SigLIP'] = ['ResNet-50 (3 seeds)', 'SigLIP probe (TTA)']
    ENS['ResNet-50 + CLIP + SigLIP'] = ['ResNet-50 (3 seeds)', 'CLIP probe (TTA)', 'SigLIP probe (TTA)']
    if BS_RUNS: ENS['all four'] = list(MEMBERS)[:2] + ['CLIP probe (TTA)', 'SigLIP probe (TTA)']
say('\nPART C. MODELS AND EQUAL-WEIGHT ENSEMBLES; museum = painters not in WikiArt train; CI = painter-level bootstrap')
rows = []
for k in ('test', 'met', 'rijks'):
    P = {n: f(k) for n, f in MEMBERS.items()}
    for e, parts in ENS.items(): P[e] = np.mean([P[p] for p in parts], 0)
    for grp in (('unseen', 'unseen, Wikidata label') if k == 'met' else ('unseen',)):
        m = mask(k, 'wikidata' if 'Wikidata' in grp else 'unseen'); y = Y[k][m]
        bs = boot(y, [P[n][m].argmax(1) for n in P], painter(k)[m], B=500)
        for j, n in enumerate(P):
            rows.append(dict(set=k, group=grp, model=n, n=int(m.sum()), **metrics(y, P[n][m]),
                             ci_low=np.percentile(bs[:, j], 2.5), ci_high=np.percentile(bs[:, j], 97.5)))
ens_tab = pd.DataFrame(rows).round(2); ens_tab.to_csv(f'{OUT}/ensembles_summary.csv', index=False)
say(ens_tab.to_string(index=False))
open(f'{OUT}/more_levers_report.txt', 'w').write('\n'.join(REPORT))
print(f'\nDone in {(time.time()-T_START)/60:.0f} min. Send back more_levers_report.txt from {OUT}')